# Hypernetwork Demo

A **hypernetwork** is a neural network that generates the weights of another neural network.

```
Task embedding z
      ↓
 [ HyperNetwork ]   ← the only thing with trainable parameters
      ↓
 Generated weights θ
      ↓
 [ TargetNetwork(x, θ) ]   ← stateless, weights injected per forward pass
      ↓
   Prediction ŷ
```

Gradients flow end-to-end: `loss → TargetNet computation → generated θ → HyperNet parameters`.

In [ ]:
import sys, math
sys.path.insert(0, '..')

import torch
import matplotlib.pyplot as plt

from src.hypernetwork.model   import HyperNetwork
from src.target_network.model import TargetNetwork
from src.utils.data           import sample_task, sample_task_batch, task_fn
from src.utils.training       import train, evaluate_task
from src.visualization.plot   import (
    plot_loss_curve, plot_task_fit,
    plot_weight_grid, plot_generalization,
)

torch.manual_seed(42)
print('PyTorch', torch.__version__)

## 1. Define the architectures

The **target network** is a tiny MLP: `1 → 32 → 32 → 1`.  
It has **no trainable parameters** — its weights come entirely from the hypernetwork.

The **hypernetwork** takes a 4-dimensional task embedding and outputs all 1,153 scalars needed by the target network in one forward pass.

In [ ]:
target_net = TargetNetwork(layer_sizes=[1, 32, 32, 1], activation='tanh')
hyper_net  = HyperNetwork(z_dim=4, hidden_dim=128, target_shapes=target_net.weight_shapes)

n_hyper  = sum(p.numel() for p in hyper_net.parameters())
n_target = target_net.n_params

print(f'Target network layer sizes : {target_net.layer_sizes}')
print(f'Target network param count : {n_target:,}')
print(f'HyperNetwork param count   : {n_hyper:,}')
print(f'\nOne HyperNet can parameterise any number of target networks')
print(f'by simply changing the input embedding z.')

## 2. Inspect a single forward pass

Let's manually run one task through the system before training, just to see the shapes.

In [ ]:
task = sample_task(freq_range=(1.0, 2.0))
print(f"Task: freq={task['freq']:.3f}  phase={task['phase']:.3f}  amp={task['amp']:.3f}")
print(f"Embedding z: {task['z'].numpy().round(3)}")

# HyperNet forward: z → list of (W, b) pairs
z = task['z'].unsqueeze(0)   # add batch dim
params = hyper_net(z)

print(f"\nGenerated parameter shapes:")
for i, (W, b) in enumerate(params):
    print(f"  Layer {i}: W={tuple(W.squeeze().shape)}  b={tuple(b.squeeze().shape)}")

# TargetNet forward: x, params → prediction
x        = torch.linspace(-math.pi, math.pi, 10).unsqueeze(-1)
tp       = [(W[0], b[0]) for W, b in params]
pred     = target_net(x, tp)
print(f"\nPrediction shape: {pred.shape}  (untrained, so random-ish values)")

## 3. Train

Each step samples a **batch of random sine tasks**, runs the full pipeline, and backpropagates through both the target network computation and the hypernetwork.

In [ ]:
history = train(
    hyper_net  = hyper_net,
    target_net = target_net,
    n_steps    = 2000,
    batch_size = 16,
    lr         = 1e-3,
    log_every  = 200,
)

In [ ]:
plot_loss_curve(history)
plt.show()

## 4. Evaluate on unseen tasks

These tasks were **never seen during training** — the hypernetwork must generalize across the task space.

In [ ]:
for _ in range(3):
    task = sample_task()
    plot_task_fit(hyper_net, target_net, task)
    plt.show()

## 5. Weight distributions

Key question: does the hypernetwork actually generate **different weights** for different tasks, or does it produce the same weights every time?

The plot below shows the distribution of generated weights for 6 randomly sampled tasks. Different tasks → genuinely different weight distributions.

In [ ]:
plot_weight_grid(hyper_net, n_tasks=6)
plt.show()

## 6. Generalization grid

A grid across (frequency × phase) space. Each cell shows predicted vs. ground-truth — illustrating how the hypernetwork smoothly interpolates across the task manifold.

In [ ]:
plot_generalization(
    hyper_net, target_net,
    freq_vals  = [0.5, 1.0, 2.0, 3.0],
    phase_vals = [0.0, math.pi/2, math.pi, 3*math.pi/2],
)
plt.show()

## 7. Interactive exploration

Manually specify a task and inspect the generated weights and prediction.

In [ ]:
import math

# ← change these!
MY_FREQ  = 1.5
MY_PHASE = math.pi / 4
MY_AMP   = 1.2

my_task = dict(
    freq=MY_FREQ, phase=MY_PHASE, amp=MY_AMP,
    z=torch.tensor([math.sin(MY_PHASE), math.cos(MY_PHASE), MY_FREQ, MY_AMP]),
)

plot_task_fit(hyper_net, target_net, my_task)
plt.show()

# Print the actual generated weight tensors
with torch.no_grad():
    params = hyper_net(my_task['z'].unsqueeze(0))
    for i, (W, b) in enumerate(params):
        print(f'Layer {i} W stats: mean={W.mean():.4f}  std={W.std():.4f}  min={W.min():.4f}  max={W.max():.4f}')